In [ ]:
# Q1
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

personalized_entries = [
    {
        "question": "how do i update my contact details",
        "answer": "Open Profile settings and edit your phone number or email.",
        "keywords": "update contact profile email phone",
        "category": "general"
    },
    {
        "question": "is there a late fee for delayed payment",
        "answer": "Yes, a small late fee may apply after the due date.",
        "keywords": "late fee penalty due payment",
        "category": "billing"
    }
]

faq_df = pd.DataFrame(fixed_entries + personalized_entries)
print(faq_df)

In [ ]:
# Q2
def rank_faq_matches(query, df):
    query_words = set(query.lower().split())
    ranked_rows = []

    for idx, row in df.iterrows():
        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        question_matches = len(query_words & question_words)
        keyword_matches = len(query_words & keyword_words)
        score = question_matches + keyword_matches

        if score > 0:
            confidence = score / max(len(query_words), 1)
            ranked_rows.append({
                "question": row["question"],
                "answer": row["answer"],
                "keywords": row["keywords"],
                "category": row["category"],
                "score": score,
                "confidence": round(confidence, 2)
            })

    ranked_df = pd.DataFrame(ranked_rows).sort_values(by=["score", "confidence"], ascending=False)
    return ranked_df.reset_index(drop=True)

sample_query = "fee payment"
q2_result = rank_faq_matches(sample_query, faq_df)
print(f"Query: {sample_query}")
print(q2_result)

In [ ]:
# Q3
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][["question", "category"]].reset_index(drop=True)

personalized_category = personalized_entries[0]["category"]
q3_result = same_category(personalized_category, faq_df)
print(f"Category used: {personalized_category}")
print(q3_result)

In [ ]:
# Q4
selected_index = 0
print("Selected question:", faq_df.loc[selected_index, "question"])

try:
    new_keyword = input("Enter a new keyword to add: " ).strip().lower()
except EOFError:
    new_keyword = "support"

if new_keyword == "":
    new_keyword = "support"

current_keywords = faq_df.loc[selected_index, "keywords"].split()
if new_keyword not in current_keywords:
    current_keywords.append(new_keyword)

faq_df.loc[selected_index, "keywords"] = " ".join(current_keywords)
faq_df.to_csv("23_faq_data.csv", index=False)
print(faq_df)

In [ ]:
# Q5
category_counts = faq_df.groupby("category").size().reset_index(name="count")
print(category_counts)

In [ ]:
# Q6
def best_match_with_ties(query, df):
    ranked_df = rank_faq_matches(query, df)
    if ranked_df.empty:
        return ranked_df

    highest_score = ranked_df["score"].max()
    top_rows = ranked_df[ranked_df["score"] == highest_score].reset_index(drop=True)
    return top_rows

tie_query = "fee"
tie_result = best_match_with_ties(tie_query, faq_df)
print(f"Query with tie: {tie_query}")
print(tie_result)

no_tie_query = "password reset"
no_tie_result = best_match_with_ties(no_tie_query, faq_df)
print(f\nQuery without tie: {no_tie_query}")
print(no_tie_result)